# BP-LLM HelpSteer2 Eval + LoRA (r=1) Fine-tuning

Evaluates BP-LLM (unary JJ) win rate on HelpSteer2 preference pairs,
then fine-tunes the policy model for 1 epoch with LoRA (r=1)
and re-evaluates Win Rate + ECE@10.

**Runtime:** Requires A100 GPU (40GB+). Use Colab Pro+ or higher.


In [1]:
!pip install peft datasets transformers accelerate tqdm -q

In [ ]:
import math
import itertools
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple, Iterable, Union

import torch
import torch.nn.functional as F
from torch.nn.functional import log_softmax
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

# =========================
# Config
# =========================
HF_TOKEN = None   # ← paste your HF token here if using gated models

# Suggested: Instruct as policy, Base as reference (Qwen2.5)
#MODEL_NAME     = "Qwen/Qwen2.5-7B-Instruct"
#REF_MODEL_NAME = "Qwen/Qwen2.5-7B"   # set to None to disable reference subtraction
#MODEL_NAME     = "meta-llama/Llama-3.2-3B-Instruct"
#REF_MODEL_NAME = "meta-llama/Llama-3.2-3B"  # Use same for reference (frozen)

MODEL_NAME     = "Qwen/Qwen2.5-7B-Instruct"
REF_MODEL_NAME = "Qwen/Qwen2.5-7B"   # set to None to disable reference subtraction



# ---- HelpSteer2 preference (pairs) ----
DATASET             = "nvidia/HelpSteer2"
PREFERENCE_SPLIT    = "train"
PREF_MIN_STRENGTH   = 1
PREF_KEEP_SPLIT     = None

# Train/Test split
TRAIN_FRAC      = 0.8
RNG_SEED        = 42

# Input limits
MAX_INPUT_TOKENS     = 1024
MAX_GEN_TOKENS       = 512
BATCH_SIZE           = 1

# Scoring
SCORING_MODE         = "mean"
LENGTH_PENALTY_ALPHA = 0.0

# Chat template toggle
USE_CHAT_TEMPLATE = True

# Length normalization: "mean" or "sum"
LENGTH_NORM = "mean"

# BP hyperparams (defaults)
BETA            = 1.0
DELTA           = 0.0
TAU             = 1.0
GAMMA           = 1.0
JJ_INNER_STEPS  = 2

# Grid tuning
DO_TUNE         = True
BETAS           = [0.5, 1.0, 2.0]
DELTAS          = ['bco', 0.0]
TAUS            = [0.5, 1.0, 2.0]
GAMMAS          = [0.5, 1.0, 1.5, 2.0]
JJ_STEPS_LIST   = [1, 2, 3, 0]        # 0 => adaptive

# LoRA training config
LORA_R           = 1
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.05
LEARNING_RATE    = 1e-4
TRAIN_BATCH_SIZE = 1  # Batch size can be increased to 8 for 3B model
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS       = 1
WARMUP_STEPS     = 100
MAX_GRAD_NORM    = 1.0
SCORE_MAX_GEN_TOKENS = 10000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Device: {DEVICE} | Dtype: {DTYPE}")


In [ ]:
# =========================
# Dataset wrapper (for DataLoader)
# =========================
class PreferenceDataset(Dataset):
    def __init__(self, records: List[Dict]):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        return self.records[idx]

def collate_fn(batch):
    return batch

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# =========================
# HelpSteer2 preference adapter
# =========================
def _safe_str(x) -> Optional[str]:
    return x if isinstance(x, str) and x.strip() else None

def _get_int(value) -> Optional[int]:
    """Best-effort parse to int (accepts floats/strings/bools)."""
    try:
        if value is None:       return None
        if isinstance(value, bool):  return int(value)
        if isinstance(value, int):   return value
        if isinstance(value, float): return int(round(value))
        s = str(value).strip()
        return int(round(float(s))) if s else None
    except Exception:
        return None

def adapt_helpsteer2_preference_row(rec: Dict) -> Optional[Dict]:
    """
    Each row: prompt, response_1, response_2, preference_strength in {-3..3}.
    s > 0  => response_2 preferred
    s < 0  => response_1 preferred
    s == 0 => tie, skip
    """
    prompt = _safe_str(rec.get("prompt") or rec.get("instruction"))
    r1 = _safe_str(
        rec.get("response_1") or rec.get("response1") or rec.get("resp1")
        or rec.get("candidate_1") or rec.get("output_1")
    )
    r2 = _safe_str(
        rec.get("response_2") or rec.get("response2") or rec.get("resp2")
        or rec.get("candidate_2") or rec.get("output_2")
    )
    s = rec.get("preference_strength") or rec.get("preference") \
        or rec.get("label") or rec.get("preference_score")
    s = _get_int(s)

    if not (prompt and r1 and r2 and (s is not None)):
        return None
    if abs(s) < int(PREF_MIN_STRENGTH):
        return None

    if s > 0:   chosen, rejected = r2, r1
    elif s < 0: chosen, rejected = r1, r2
    else:       return None

    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

def load_hs2_preference_pairs() -> List[Dict]:
    """
    Load HelpSteer2 preference split and adapt to (prompt, chosen, rejected).
    Optionally filters by the internal 'split' column if PREF_KEEP_SPLIT is set.
    """
    ds = load_dataset(DATASET, data_dir="preference", split=PREFERENCE_SPLIT)
    adapted: List[Dict] = []
    for rec in ds:
        if PREF_KEEP_SPLIT is not None:
            if rec.get("split") != PREF_KEEP_SPLIT:
                continue
        a = adapt_helpsteer2_preference_row(rec)
        if a is not None:
            adapted.append(a)
    return adapted


In [ ]:
# =========================
# Prompt rendering
# =========================
def render_prompt_with_policy_template(
    policy_tokenizer,
    prompt_or_messages: Union[str, List[Dict[str,str]]]
) -> str:
    if USE_CHAT_TEMPLATE and hasattr(policy_tokenizer, "apply_chat_template"):
        try:
            if isinstance(prompt_or_messages, list):
                msgs = prompt_or_messages
            else:
                msgs = [{"role": "user", "content": str(prompt_or_messages).strip()}]
            return policy_tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            pass
    if isinstance(prompt_or_messages, list):
        p = "\n".join(f"{m.get('role','user').upper()}: {(m.get('content') or '').strip()}" for m in prompt_or_messages)
    else:
        p = str(prompt_or_messages).strip()
    return p if p.endswith("\n") else (p + "\n")

def render_prompt_list(policy_tokenizer, prompts: List[Union[str, List[Dict[str,str]]]]) -> List[str]:
    return [render_prompt_with_policy_template(policy_tokenizer, p) for p in prompts]

def concat_prompt_response_text(prompt_text: str, response: str) -> Tuple[str, str]:
    full_text = prompt_text + ("" if prompt_text.endswith("\n") else "\n") + str(response).strip()
    return full_text, prompt_text


In [ ]:
# =========================
# Tokenization / Scoring
# =========================
@torch.no_grad()
def sequence_logprob_list(
    model,
    tokenizer,
    prompt_texts: List[str],
    responses: List[str],
    length_norm: str = LENGTH_NORM,
) -> List[Optional[float]]:
    """
    Returns mean/sum log-prob for each (prompt, response). If response was truncated
    (i.e., prompt consumed the full sequence), returns None for that item.
    """
    vals: List[Optional[float]] = []
    for i in range(0, len(prompt_texts), BATCH_SIZE):
        batch_prompts = prompt_texts[i:i+BATCH_SIZE]
        batch_resps   = responses[i:i+BATCH_SIZE]

        batch_inputs, batch_prompt_lens = [], []
        for p_txt, r in zip(batch_prompts, batch_resps):
            full_text, prompt_only = concat_prompt_response_text(p_txt, r)

            toks = tokenizer(
                full_text,
                truncation=True,
                max_length=min(MAX_INPUT_TOKENS + MAX_GEN_TOKENS, tokenizer.model_max_length),
                return_tensors="pt",
                add_special_tokens=True,
            )
            tok_prompt = tokenizer(
                prompt_only,
                truncation=True,
                max_length=MAX_INPUT_TOKENS,
                return_tensors="pt",
                add_special_tokens=True,
            )
            p_len = tok_prompt["input_ids"].shape[-1]
            batch_inputs.append(toks)
            batch_prompt_lens.append(p_len)

        pad_id = tokenizer.pad_token_id or 0
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [bi["input_ids"].squeeze(0) for bi in batch_inputs],
            batch_first=True, padding_value=pad_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [bi["attention_mask"].squeeze(0) for bi in batch_inputs],
            batch_first=True, padding_value=0
        )
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        logprobs = log_softmax(logits.to(torch.float32), dim=-1)

        for b in range(input_ids.size(0)):
            p_len = batch_prompt_lens[b]
            ids   = input_ids[b]
            masks = attention_mask[b]
            L     = int(masks.sum().item())
            if p_len >= L:
                vals.append(None)
                continue
            targ = ids[p_len:L]
            pred = logprobs[b, p_len-1:L-1]
            lp = pred.gather(-1, targ.unsqueeze(-1)).squeeze(-1)
            score = float((lp.mean() if length_norm == "mean" else lp.sum()).cpu())
            vals.append(score)
    return vals


def get_logps_with_grad(
    model,
    tokenizer,
    prompt_texts: List[str],
    responses: List[str],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Like sequence_logprob_list but RETAINS the computation graph for backprop.
    Returns (logps [N], valid_mask [N]).  logps lives on DEVICE with grad.
    """
    batch_logps = []
    batch_valid = []
    max_length  = MAX_INPUT_TOKENS + SCORE_MAX_GEN_TOKENS

    for p_txt, resp in zip(prompt_texts, responses):
        full_text = p_txt + resp.strip()

        toks_full   = tokenizer(full_text,  truncation=True, max_length=max_length,
                                return_tensors="pt", add_special_tokens=True)
        toks_prompt = tokenizer(p_txt,      truncation=True, max_length=MAX_INPUT_TOKENS,
                                return_tensors="pt", add_special_tokens=True)

        input_ids      = toks_full["input_ids"].to(DEVICE)
        attention_mask = toks_full["attention_mask"].to(DEVICE)
        p_len          = toks_prompt["input_ids"].shape[-1]

        with torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

        logprobs = log_softmax(logits.float(), dim=-1)
        seq_len  = int(attention_mask.sum().item())

        if p_len >= seq_len:
            batch_logps.append(torch.tensor(0.0, device=DEVICE))
            batch_valid.append(False)
            continue

        end            = min(seq_len, p_len + SCORE_MAX_GEN_TOKENS)
        response_ids   = input_ids[0, p_len:end]
        response_logps = logprobs[0, p_len-1:end-1]
        selected       = response_logps.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)

        logp = selected.mean() if LENGTH_NORM == "mean" else selected.sum()
        batch_logps.append(logp)       # stays on DEVICE, retains grad
        batch_valid.append(True)

    return torch.stack(batch_logps), torch.tensor(batch_valid)


In [ ]:
# =========================
# Model loaders
# =========================
def load_causal_lm(model_id: str, token: Optional[str], dtype=DTYPE, device=DEVICE):
    """Plain frozen loader (used for reference model)."""
    tok = AutoTokenizer.from_pretrained(
        model_id, token=token, use_fast=True, trust_remote_code=True
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=token, torch_dtype=dtype, trust_remote_code=True
    ).to(device)
    model.eval()
    return tok, model


def load_policy_with_lora(model_id: str, token: Optional[str]):
    """Load policy model and attach LoRA adapters (r=LORA_R)."""
    print(f"Loading tokenizer from {model_id}...")
    tok = AutoTokenizer.from_pretrained(
        model_id, token=token, use_fast=True, trust_remote_code=True
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    print(f"Loading base model from {model_id}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=token, torch_dtype=DTYPE,
        trust_remote_code=True, device_map="auto"
    )

    print(f"Adding LoRA adapters (r={LORA_R}, alpha={LORA_ALPHA})...")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return tok, model


In [ ]:
# =========================
# BP-LLM (unary JJ) posterior
# =========================
@dataclass
class BPParams:
    beta: float = BETA
    delta: float = DELTA
    tau: float = TAU
    gamma: float = GAMMA
    jj_steps: int = JJ_INNER_STEPS

def jj_lambda(xi: float) -> float:
    if xi < 1e-8:
        return 1.0 / 8.0
    return math.tanh(xi / 2.0) / (4.0 * xi)

def bp_unary_posterior(mu_prior: float, b: int, params: BPParams) -> Tuple[float, float]:
    tau2 = params.tau ** 2
    gamma_tilde = params.gamma * (2 * b - 1)
    mu_hat = mu_prior
    tau2_hat = tau2
    for _ in range(params.jj_steps):
        xi = abs(params.gamma) * math.sqrt(mu_hat * mu_hat + tau2_hat)
        lam = jj_lambda(xi)
        Lambda = (1.0 / tau2) + 2.0 * lam
        eta    = (mu_prior / tau2) + 0.5 * gamma_tilde
        mu_hat = eta / Lambda
        tau2_hat = 1.0 / Lambda
    return mu_hat, tau2_hat

def bp_unary_posterior_adaptive(mu_prior: float, b: int, params: BPParams,
                                tol: float = 1e-4, max_steps: int = 50) -> Tuple[float, float]:
    tau2 = params.tau ** 2
    gamma_tilde = params.gamma * (2 * b - 1)
    mu_hat = mu_prior
    tau2_hat = tau2
    steps = params.jj_steps if params.jj_steps and params.jj_steps > 0 else max_steps
    for _ in range(steps):
        mu_prev = mu_hat
        xi = abs(params.gamma) * math.sqrt(mu_hat * mu_hat + tau2_hat)
        lam = jj_lambda(xi)
        Lambda = (1.0 / tau2) + 2.0 * lam
        eta    = (mu_prior / tau2) + 0.5 * gamma_tilde
        mu_hat = eta / Lambda
        tau2_hat = 1.0 / Lambda
        if (params.jj_steps is None or params.jj_steps <= 0) and abs(mu_hat - mu_prev) < tol:
            break
    return mu_hat, tau2_hat


In [ ]:
# =========================
# Caching scores (pairwise)
# =========================
@torch.no_grad()
def cache_pairwise_scores(
    records: List[Dict],
    policy_model,
    ref_model,
    policy_tok,
    ref_tok=None
) -> Tuple[torch.Tensor, int]:
    """
    Returns base_s tensor shaped [2N] where first N are chosen scores, next N are rejected.
    Each score is (log pi - log pi_ref) with length normalization.
    """
    prompts  = [r["prompt"]   for r in records]
    resps_ch = [r["chosen"]   for r in records]
    resps_rj = [r["rejected"] for r in records]

    prompt_texts = render_prompt_list(policy_tok, prompts)

    pol_ch = sequence_logprob_list(policy_model, policy_tok, prompt_texts, resps_ch, LENGTH_NORM)
    pol_rj = sequence_logprob_list(policy_model, policy_tok, prompt_texts, resps_rj, LENGTH_NORM)

    if ref_model is None:
        ref_ch = [0.0 if (s is not None) else None for s in pol_ch]
        ref_rj = [0.0 if (s is not None) else None for s in pol_rj]
    else:
        if ref_tok is None:
            raise ValueError("ref_model provided without ref_tok")
        ref_ch = sequence_logprob_list(ref_model, ref_tok, prompt_texts, resps_ch, LENGTH_NORM)
        ref_rj = sequence_logprob_list(ref_model, ref_tok, prompt_texts, resps_rj, LENGTH_NORM)

    chosen_vals, rejected_vals = [], []
    for sc, sr, rc, rr in zip(pol_ch, pol_rj, ref_ch, ref_rj):
        if (sc is None) or (sr is None) or (rc is None) or (rr is None):
            continue
        chosen_vals.append(sc - rc)
        rejected_vals.append(sr - rr)

    if len(chosen_vals) == 0:
        raise RuntimeError("No valid pairs after scoring. "
                           "Consider increasing MAX_*_TOKENS or using LENGTH_NORM='sum'.")

    base_s = torch.tensor(chosen_vals + rejected_vals, dtype=torch.float32)
    N = len(chosen_vals)
    return base_s, N


In [ ]:
# =========================
# Evaluation & tuning (ECE + grid search)
# =========================
def _phi_standard_normal(z: float) -> float:
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def pairwise_win_probs_from_priors(base_s: torch.Tensor, N: int, params: BPParams) -> List[float]:
    diffs = (params.beta * (base_s[:N] - base_s[N:])).detach().cpu().tolist()
    tau = float(params.tau)
    if tau <= 1e-12:
        return [1.0 if d > 0 else 0.0 for d in diffs]
    denom = math.sqrt(2.0) * tau
    return [_phi_standard_normal(d / denom) for d in diffs]

def ece_from_probs(probs: List[float], labels: List[int], n_bins: int = 10) -> float:
    p = torch.tensor(probs, dtype=torch.float32).clamp(0.0, 1.0)
    y = torch.tensor(labels, dtype=torch.int64)
    pred = (p >= 0.5).to(torch.int64)
    conf = torch.where(pred == 1, p, 1.0 - p)

    ece = 0.0
    bin_edges = torch.linspace(0.0, 1.0, n_bins + 1)
    for b in range(n_bins):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        in_bin = (conf >= lo) & (conf <= hi) if b == 0 else (conf > lo) & (conf <= hi)
        n = int(in_bin.sum().item())
        if n == 0:
            continue
        acc_bin  = (pred[in_bin] == y[in_bin]).float().mean().item()
        conf_bin = conf[in_bin].float().mean().item()
        ece += (n / len(probs)) * abs(acc_bin - conf_bin)
    return float(ece)

def ece10_pairwise(base_s: torch.Tensor, N: int, params: BPParams, symmetric: bool = True) -> float:
    probs_pos = pairwise_win_probs_from_priors(base_s, N, params)
    if symmetric:
        probs, labels = [], []
        for p in probs_pos:
            probs.append(p);       labels.append(1)
            probs.append(1.0 - p); labels.append(0)
    else:
        probs, labels = probs_pos, [1] * len(probs_pos)
    return ece_from_probs(probs, labels, n_bins=10)

def ece_bin_details(probs: List[float], labels: List[int], n_bins: int = 10) -> Tuple[float, List[Tuple[int, float, float, float, int]]]:
    p = torch.tensor(probs, dtype=torch.float32).clamp(0.0, 1.0)
    y = torch.tensor(labels, dtype=torch.int64)
    pred = (p >= 0.5).to(torch.int64)
    conf = torch.where(pred == 1, p, 1.0 - p)

    ece  = 0.0
    rows = []
    bin_edges = torch.linspace(0.0, 1.0, n_bins + 1)
    for b in range(n_bins):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        in_bin = (conf >= lo) & (conf <= hi) if b == 0 else (conf > lo) & (conf <= hi)
        n = int(in_bin.sum().item())
        if n == 0:
            continue
        acc_bin  = (pred[in_bin] == y[in_bin]).float().mean().item()
        conf_bin = conf[in_bin].float().mean().item()
        gap = abs(acc_bin - conf_bin)
        ece += (n / len(probs)) * gap
        rows.append((b + 1, conf_bin, acc_bin, gap, n))
    return float(ece), rows

def print_ece_table(rows: List[Tuple[int, float, float, float, int]], title: str = ""):
    if title:
        print(f"\n[{title}]")
    print("  Bin   Confidence   Accuracy     Gap          Count")
    print("  ----- ------------ ------------ ------------ --------")
    for (b, conf, acc, gap, n) in rows:
        print(f"  {b:<5d} {conf:>12.4f} {acc:>12.4f} {gap:>12.4f} {n:>8d}")

def ece_details_pairwise(base_s, N, params, symmetric=True, n_bins=10):
    probs_pos = pairwise_win_probs_from_priors(base_s, N, params)
    if symmetric:
        probs, labels = [], []
        for p in probs_pos:
            probs.append(p);       labels.append(1)
            probs.append(1.0 - p); labels.append(0)
    else:
        probs, labels = probs_pos, [1] * len(probs_pos)
    return ece_bin_details(probs, labels, n_bins=n_bins)

def estimate_delta_bco(base_s: torch.Tensor, beta: float, N: int) -> float:
    r_pos = beta * base_s[:N]
    r_neg = beta * base_s[N:]
    return 0.5 * (float(r_pos.mean()) + float(r_neg.mean()))

def evaluate_with_cached(
    base_s: torch.Tensor, N: int, params: BPParams,
    use_adaptive_jj: bool = True, use_labels: bool = True
) -> float:
    mu = (params.beta * base_s - params.delta).tolist()
    correct = 0
    for i in range(N):
        mu_w_prior = mu[i]
        mu_l_prior = mu[i + N]
        if not use_labels:
            if mu_w_prior > mu_l_prior:
                correct += 1
            continue
        if use_adaptive_jj:
            mu_w_post, _ = bp_unary_posterior_adaptive(mu_w_prior, b=1, params=params)
            mu_l_post, _ = bp_unary_posterior_adaptive(mu_l_prior, b=0, params=params)
        else:
            mu_w_post, _ = bp_unary_posterior(mu_w_prior, b=1, params=params)
            mu_l_post, _ = bp_unary_posterior(mu_l_prior, b=0, params=params)
        if mu_w_post > mu_l_post:
            correct += 1
    return 100.0 * correct / N

def binom_ci_95(pct: float, N: int) -> Tuple[float, float]:
    p = pct / 100.0
    se = math.sqrt(p * (1 - p) / max(N, 1))
    return max(0.0, 100.0 * (p - 1.96 * se)), min(100.0, 100.0 * (p + 1.96 * se))

def grid_search(
    records, policy_model, ref_model, policy_tok, ref_tok,
    betas, deltas, taus, gammas, jj_steps_list,
    use_adaptive_jj=True, estimate_delta=True, split_name="TRAIN"
):
    base_s, N = cache_pairwise_scores(records, policy_model, ref_model, policy_tok, ref_tok)
    tried = []
    best  = ((-1e9, -1e9), None)

    for beta, delta_spec, tau, gamma, jj_steps in itertools.product(
        betas, deltas, taus, gammas, jj_steps_list
    ):
        if estimate_delta and (delta_spec == "bco"):
            delta    = estimate_delta_bco(base_s, beta, N)
            delta_tag = "bco"
        else:
            delta    = float(delta_spec)
            delta_tag = f"{delta:.4f}"

        infer_params = BPParams(beta=beta, delta=delta, tau=tau, gamma=0.0, jj_steps=0)
        wr_inf  = evaluate_with_cached(base_s, N, infer_params, use_adaptive_jj=False, use_labels=False)
        ece_inf = ece10_pairwise(base_s, N, infer_params, symmetric=True)
        lo, hi  = binom_ci_95(wr_inf, N)

        dbg_params = BPParams(beta=beta, delta=delta, tau=tau, gamma=gamma, jj_steps=jj_steps)
        wr_lbl = evaluate_with_cached(base_s, N, dbg_params, use_adaptive_jj=use_adaptive_jj, use_labels=True)

        tried.append((wr_inf, ece_inf, lo, hi, infer_params, dbg_params))

        score = (wr_inf, -ece_inf)
        if (best[1] is None) or (score > best[0]):
            best = (score, infer_params)

        print(
            f"[TUNE/{split_name}] WR_inf={wr_inf:.2f}% [{lo:.1f}, {hi:.1f}] | "
            f"ECE@10={ece_inf:.4f} (sym) | "
            f"(dbg) WR_lbl={wr_lbl:.2f}% | "
            f"beta={beta} delta={delta_tag} tau={tau} "
            f"(dbg gamma={gamma} jj_steps={jj_steps})"
        )

    (best_wr_inf, neg_best_ece), best_infer = best
    best_ece = -neg_best_ece
    lo, hi = binom_ci_95(best_wr_inf, N)
    print(
        f"\n[BP-LLM Tuning ({split_name})] Best (inference) "
        f"WR_inf={best_wr_inf:.2f}% [{lo:.1f}, {hi:.1f}] | "
        f"ECE@10={best_ece:.4f} (sym) with "
        f"beta={best_infer.beta} delta={best_infer.delta:.4f} tau={best_infer.tau} "
        f"gamma={best_infer.gamma} jj_steps={best_infer.jj_steps}"
    )
    return tried, best_infer


In [ ]:
def train_one_epoch(policy_model, ref_model, tokenizer, train_records, best_params):
    """
    Train policy for 1 epoch with LoRA using softplus margin loss on BP priors.
    Loss = softplus(-(mu_chosen - mu_rejected))
    where mu = beta * (log pi - log pi_ref) - delta
    """
    policy_model.train()

    # ─── PRE-CACHE all reference scores (ref is already on GPU here) ───
    print("Pre-caching reference model scores...")
    ref_model.eval()
    ref_cache = {}  # key: index → (ref_ch, ref_rj, valid)
    prompt_texts_all = [render_prompt_with_policy_template(tokenizer, r["prompt"]) for r in train_records]
    chosen_all   = [r["chosen"]   for r in train_records]
    rejected_all = [r["rejected"] for r in train_records]

    with torch.no_grad():
        ref_ch_all = sequence_logprob_list(ref_model, tokenizer, prompt_texts_all, chosen_all,   LENGTH_NORM)
        ref_rj_all = sequence_logprob_list(ref_model, tokenizer, prompt_texts_all, rejected_all, LENGTH_NORM)

    for i in range(len(train_records)):
        ref_cache[i] = (ref_ch_all[i], ref_rj_all[i])

    del ref_ch_all, ref_rj_all
    print(f"Cached {len(ref_cache)} reference scores. Offloading ref model to CPU...")

    # ─── Offload ref to CPU — no longer needed on GPU ───
    ref_model.cpu()
    clear_memory()

    # ─── Training setup ───
    # We need a dataloader that preserves original indices for cache lookup
    indexed_records = list(enumerate(train_records))
    random.shuffle(indexed_records)

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=LEARNING_RATE)
    num_training_steps = len(train_records) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=max(num_training_steps, 1)
    )

    print(f"\n{'='*80}")
    print(f"LoRA Training — 1 epoch | batch={TRAIN_BATCH_SIZE} | accum={GRAD_ACCUM_STEPS} "
          f"| effective_batch={TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS} | steps={num_training_steps}")
    print(f"BP params used: beta={best_params.beta} delta={best_params.delta:.4f} tau={best_params.tau}")
    print(f"{'='*80}")

    total_loss, total_margin, total_wr, num_batches = 0.0, 0.0, 0.0, 0

    # Manual batching to preserve indices
    pbar = tqdm(range(0, len(indexed_records), TRAIN_BATCH_SIZE), desc="Epoch 1")

    for batch_idx, start in enumerate(pbar):
        batch = indexed_records[start:start + TRAIN_BATCH_SIZE]
        indices  = [idx for idx, _   in batch]
        prompts  = [rec["prompt"]   for _, rec in batch]
        chosen   = [rec["chosen"]   for _, rec in batch]
        rejected = [rec["rejected"] for _, rec in batch]

        prompt_texts = [render_prompt_with_policy_template(tokenizer, p) for p in prompts]

        # Policy log-probs WITH grad (on GPU)
        pol_ch, valid_ch = get_logps_with_grad(policy_model, tokenizer, prompt_texts, chosen)
        pol_rj, valid_rj = get_logps_with_grad(policy_model, tokenizer, prompt_texts, rejected)

        # Reference log-probs from CACHE (no GPU transfer needed)
        ref_ch_vals, ref_rj_vals, valid_ref_list = [], [], []
        for i in indices:
            rc, rr = ref_cache[i]
            ref_ch_vals.append(rc if rc is not None else 0.0)
            ref_rj_vals.append(rr if rr is not None else 0.0)
            valid_ref_list.append(rc is not None and rr is not None)

        ref_ch    = torch.tensor(ref_ch_vals,    device=DEVICE)
        ref_rj    = torch.tensor(ref_rj_vals,    device=DEVICE)
        valid_ref = torch.tensor(valid_ref_list)

        # Valid = both policy and ref scored successfully
        valid = valid_ch & valid_rj & valid_ref
        if not valid.any():
            continue

        # BP priors: mu = beta * (log pi - log pi_ref) - delta
        mu_ch = best_params.beta * (pol_ch[valid] - ref_ch[valid]) - best_params.delta
        mu_rj = best_params.beta * (pol_rj[valid] - ref_rj[valid]) - best_params.delta

        margin = mu_ch - mu_rj                          # want > 0
        loss   = F.softplus(-margin).mean()             # softplus margin loss
        loss   = loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # Logging (detached)
        with torch.no_grad():
            m = margin.detach()
            total_loss   += (loss.item() * GRAD_ACCUM_STEPS)
            total_margin += m.mean().item()
            total_wr     += (m > 0).float().mean().item() * 100
            num_batches  += 1

        pbar.set_postfix({
            'loss':   f"{loss.item() * GRAD_ACCUM_STEPS:.4f}",
            'margin': f"{m.mean().item():.4f}",
            'wr':     f"{(m > 0).float().mean().item()*100:.1f}%",
            'lr':     f"{scheduler.get_last_lr()[0]:.2e}"
        })

    # Handle leftover accumulation steps
    total_batches = len(range(0, len(indexed_records), TRAIN_BATCH_SIZE))
    if total_batches % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_loss   = total_loss   / max(num_batches, 1)
    avg_margin = total_margin / max(num_batches, 1)
    avg_wr     = total_wr     / max(num_batches, 1)

    print(f"\n  Train avg — Loss: {avg_loss:.4f} | Margin: {avg_margin:.4f} | WR: {avg_wr:.1f}%")
    return policy_model

In [ ]:
# =========================
# Main — runs the full pipeline
# =========================

# ---------- Load models ----------
print("Loading policy model (with LoRA)...")
tok_policy, policy = load_policy_with_lora(MODEL_NAME, HF_TOKEN)

tok_ref, ref = None, None
if REF_MODEL_NAME:
    print("Loading reference model (frozen)...")
    tok_ref, ref = load_causal_lm(REF_MODEL_NAME, HF_TOKEN)

# ---------- Dataset ----------
print(f"\nLoading HelpSteer2 preference pairs …")
adapted = load_hs2_preference_pairs()
print(f"Adapted pairs (|strength| >= {PREF_MIN_STRENGTH}"
      f"{' & split=' + str(PREF_KEEP_SPLIT) if PREF_KEEP_SPLIT else ''}): {len(adapted)}")
if not adapted:
    raise RuntimeError(
        "No preference pairs found. Check data_dir='preference' and your dataset version."
    )

# Sanity preview
ex = adapted[0]
print("Example:", {k: (ex[k][:100]+"…") if isinstance(ex.get(k), str) and len(ex[k])>100 else ex.get(k)
                   for k in ("prompt","chosen","rejected")})

random.Random(RNG_SEED).shuffle(adapted)
n_all   = len(adapted)
n_train = max(1, int(TRAIN_FRAC * n_all))
train_records = adapted[:n_train]
test_records  = adapted[n_train:]
print(f"Train records: {len(train_records)} | Test records: {len(test_records)}")

# ---------- Sanity checks on TRAIN (pre-training) ----------
base_s_tr, N_tr = cache_pairwise_scores(train_records, policy, ref, tok_policy, tok_ref)

params0_lbl = BPParams(beta=BETA, delta=DELTA, tau=TAU, gamma=GAMMA, jj_steps=JJ_INNER_STEPS)
wr0_lbl = evaluate_with_cached(base_s_tr, N_tr, params0_lbl, use_adaptive_jj=False, use_labels=True)
lo0_lbl, hi0_lbl = binom_ci_95(wr0_lbl, N_tr)
print(
    f"[Sanity TRAIN (label-cond)] WR={wr0_lbl:.2f}% [{lo0_lbl:.1f}, {hi0_lbl:.1f}] on {N_tr} pairs "
    f"| beta={params0_lbl.beta} delta={params0_lbl.delta} tau={params0_lbl.tau} "
    f"gamma={params0_lbl.gamma} jj_steps={params0_lbl.jj_steps}"
)

params0_inf = BPParams(beta=BETA, delta=DELTA, tau=TAU, gamma=0.0, jj_steps=0)
wr0_inf = evaluate_with_cached(base_s_tr, N_tr, params0_inf, use_adaptive_jj=False, use_labels=False)
lo0_inf, hi0_inf = binom_ci_95(wr0_inf, N_tr)
ece_tr0, rows_tr0 = ece_details_pairwise(base_s_tr, N_tr, params0_inf, symmetric=True, n_bins=10)
print(
    f"[Sanity TRAIN (inference)] WR={wr0_inf:.2f}% [{lo0_inf:.1f}, {hi0_inf:.1f}] | "
    f"ECE@10={ece_tr0:.4f} (sym) | beta={params0_inf.beta} delta={params0_inf.delta:.4f} tau={params0_inf.tau}"
)

# ---------- Grid tune on TRAIN ----------
if DO_TUNE:
    print("\nTuning on TRAIN (selecting by inference-time WR + ECE@10)...")
    _tries, best_infer = grid_search(
        train_records, policy, ref, tok_policy, tok_ref,
        betas=BETAS, deltas=DELTAS, taus=TAUS, gammas=GAMMAS,
        jj_steps_list=JJ_STEPS_LIST, use_adaptive_jj=True, estimate_delta=True, split_name="TRAIN"
    )
else:
    best_infer = params0_inf

# ---------- Pre-training TEST eval ----------
print(f"\n{'='*80}")
print("PRE-TRAINING TEST evaluation...")
print(f"{'='*80}")
base_s_te_pre, N_te = cache_pairwise_scores(test_records, policy, ref, tok_policy, tok_ref)
test_params = BPParams(
    beta=best_infer.beta, delta=best_infer.delta,
    tau=best_infer.tau, gamma=0.0, jj_steps=0
)
wr_te_pre = evaluate_with_cached(base_s_te_pre, N_te, test_params, use_adaptive_jj=False, use_labels=False)
lo_pre, hi_pre = binom_ci_95(wr_te_pre, N_te)
ece_te_pre, rows_pre = ece_details_pairwise(base_s_te_pre, N_te, test_params, symmetric=True, n_bins=10)
print(
    f"[PRE-TRAIN TEST] WR={wr_te_pre:.2f}% [{lo_pre:.1f}, {hi_pre:.1f}] | "
    f"ECE@10={ece_te_pre:.4f} (sym) | N={N_te}"
)
print_ece_table(rows_pre, title="ECE Calibration Details (TEST — pre-training)")



In [ ]:
# ---------- LoRA Training (1 epoch) ----------
# Offload ref to CPU during training to save GPU memory, then move back
ref.cuda()         # ← move back to GPU BEFORE calling train
clear_memory()


policy = train_one_epoch(policy, ref, tok_policy, train_records, best_infer)

ref.cuda()
clear_memory()

# ---------- Post-training TEST eval ----------
print(f"\n{'='*80}")
print("POST-TRAINING TEST evaluation...")
print(f"{'='*80}")
base_s_te_post, N_te_post = cache_pairwise_scores(test_records, policy, ref, tok_policy, tok_ref)
wr_te_post = evaluate_with_cached(base_s_te_post, N_te_post, test_params, use_adaptive_jj=False, use_labels=False)
lo_post, hi_post = binom_ci_95(wr_te_post, N_te_post)
ece_te_post, rows_post = ece_details_pairwise(base_s_te_post, N_te_post, test_params, symmetric=True, n_bins=10)
print(
    f"[POST-TRAIN TEST] WR={wr_te_post:.2f}% [{lo_post:.1f}, {hi_post:.1f}] | "
    f"ECE@10={ece_te_post:.4f} (sym) | N={N_te_post}"
)
print_ece_table(rows_post, title="ECE Calibration Details (TEST — post-training)")

# ---------- Summary ----------
print(f"\n{'='*80}")
print("SUMMARY (TEST set)")
print(f"{'='*80}")
print(f"  {'Metric':<25} {'Pre-train':>12} {'Post-train':>12} {'Delta':>12}")
print(f"  {'-'*25} {'-'*12} {'-'*12} {'-'*12}")
for label, pre, post in [
    ("Win Rate (%)",     wr_te_pre,  wr_te_post),
    ("ECE@10 (sym)",     ece_te_pre, ece_te_post),
]:
    delta = post - pre
    sign  = "+" if delta >= 0 else ""
    print(f"  {label:<25} {pre:>12.4f} {post:>12.4f} {sign}{delta:>11.4f}")
print(f"  {'BP params':<25}   beta={test_params.beta}  delta={test_params.delta:.4f}  tau={test_params.tau}")
print(f"{'='*80}")


In [ ]:
from google.colab import runtime
runtime.unassign()  # disconnects + releases the VM
